In [ ]:
overwrite_previous_preprocessed_data = False

In [2]:
import datetime

figi_to_date_ranges = {
    'GT': [(datetime.date(1991, 5, 6), datetime.date(1999, 10, 31))],
    'CVX': [(datetime.date(1991, 5, 6), datetime.date(1999, 10, 31)), (datetime.date(2008, 2, 19), datetime.date(2025, 8, 1))],
    'IP': [(datetime.date(1991, 5, 6), datetime.date(2004, 4, 7))],
    'MO': [(datetime.date(1991, 5, 6), datetime.date(2008, 2, 18))],
    'HON': [(datetime.date(1991, 5, 6), datetime.date(2008, 2, 18)), (datetime.date(2020, 8, 31), datetime.date(2025, 8, 1))],
    'AIG': [(datetime.date(2004, 4, 8), datetime.date(2008, 9, 21))],
    'C': [(datetime.date(1998, 10, 8), datetime.date(2009, 6, 7))],
    'AA': [(datetime.date(1991, 5, 6), datetime.date(2013, 9, 22))],
    'BAC': [(datetime.date(2008, 2, 19), datetime.date(2013, 9, 22))],
    'HPQ': [(datetime.date(1997, 3, 17), datetime.date(2013, 9, 22))],
    'T': [(datetime.date(2005, 11, 18), datetime.date(2015, 3, 18))],
    'DD': [(datetime.date(1991, 5, 6), datetime.date(2017, 8, 31))],
    'GE': [(datetime.date(1991, 5, 6), datetime.date(2018, 6, 25))],
    'XOM': [(datetime.date(1991, 5, 6), datetime.date(2020, 8, 30))],
    'PFE': [(datetime.date(2004, 4, 8), datetime.date(2020, 8, 30))],
    'RTX': [(datetime.date(2020, 4, 6), datetime.date(2020, 8, 30))],
    'INTC': [(datetime.date(1999, 11, 1), datetime.date(2024, 11, 7))],
    'DOW': [(datetime.date(2019, 4, 2), datetime.date(2024, 11, 7))],
    'AXP': [(datetime.date(1991, 5, 6), datetime.date(2025, 8, 1))],
    'BA': [(datetime.date(1991, 5, 6), datetime.date(2025, 8, 1))],
    'CAT': [(datetime.date(1991, 5, 6), datetime.date(2025, 8, 1))],
    'KO': [(datetime.date(1991, 5, 6), datetime.date(2025, 8, 1))],
    'IBM': [(datetime.date(1991, 5, 6), datetime.date(2025, 8, 1))],
    'JPM': [(datetime.date(1991, 5, 6), datetime.date(2025, 8, 1))],
    'MCD': [(datetime.date(1991, 5, 6), datetime.date(2025, 8, 1))],
    'MRK': [(datetime.date(1991, 5, 6), datetime.date(2025, 8, 1))],
    'MMM': [(datetime.date(1991, 5, 6), datetime.date(2025, 8, 1))],
    'PG': [(datetime.date(1991, 5, 6), datetime.date(2025, 8, 1))],
    'DIS': [(datetime.date(1991, 5, 6), datetime.date(2025, 8, 1))],
    'JNJ': [(datetime.date(1997, 3, 17), datetime.date(2025, 8, 1))],
    'WMT': [(datetime.date(1997, 3, 17), datetime.date(2025, 8, 1))],
    'MSFT': [(datetime.date(1999, 11, 1), datetime.date(2025, 8, 1))],
    'HD': [(datetime.date(1999, 11, 1), datetime.date(2025, 8, 1))],
    'VZ': [(datetime.date(2004, 4, 8), datetime.date(2025, 8, 1))],
    'TRV': [(datetime.date(2009, 6, 8), datetime.date(2025, 8, 1))],
    'CSCO': [(datetime.date(2009, 6, 8), datetime.date(2025, 8, 1))],
    'UNH': [(datetime.date(2012, 9, 24), datetime.date(2025, 8, 1))],
    'GS': [(datetime.date(2013, 9, 23), datetime.date(2025, 8, 1))],
    'NKE': [(datetime.date(2013, 9, 23), datetime.date(2025, 8, 1))],
    'V': [(datetime.date(2013, 9, 23), datetime.date(2025, 8, 1))],
    'AAPL': [(datetime.date(2015, 3, 19), datetime.date(2025, 8, 1))],
    'AMGN': [(datetime.date(2020, 8, 31), datetime.date(2025, 8, 1))],
    'CRM': [(datetime.date(2020, 8, 31), datetime.date(2025, 8, 1))],
    'AMZN': [(datetime.date(2024, 2, 26), datetime.date(2025, 8, 1))],
    'NVDA': [(datetime.date(2024, 11, 8), datetime.date(2025, 8, 1))],
    'SHW': [(datetime.date(2024, 11, 8), datetime.date(2025, 8, 1))]
}

<!-- 1. Make a data structure holding, for each date from 10/01/97 (MM/YY/DD) to 08/01/25 inclusive, an empty set.
2. For each base ticker in constituents:
    1. For each FIGI in this base ticker's value:
        1. For each object in this FIGI's value:
            1. Take date_range_inclusive.
            2. If the first date is '<10/01/97', simply treat it as '10/01/97'.
            3. For each date between the first date and the last date inclusive:
                1. Take the corresponding set from the data structure mentioned above.
                2. Add the FIGI to that set. -->

In [3]:
from datetime import date, datetime, timedelta
from collections import Counter

rebalance_period_weeks = 13

start = date(2008, 2, 20)   # First date on or after 19th February 2008 that is an even number of weeks before 30th April 2025.
end = date(2025, 4, 30) # Last rebalance date for which there are 13 Wednesdays of data ahead

djia_daily_constituent_figis = {}

current_date = start
while current_date <= end:
    djia_daily_constituent_figis[current_date] = set()
    current_date += timedelta(1)

test = []

for figi, inclusive_date_ranges in figi_to_date_ranges.items():
    for date_range_inclusive in inclusive_date_ranges:
        first_date, last_date = date_range_inclusive

        current_date = max(first_date, start)
        last_date = min(last_date, end)
        while current_date <= last_date:
            djia_daily_constituent_figis[current_date].add(figi)
            current_date += timedelta(1)

for date, figi_list in djia_daily_constituent_figis.items():
    test.append(len(figi_list))
    if len(test) > 2 and test[-1] > test[-2]:
        print(date)

print(Counter(test))    

print(test)

2009-06-08
2012-09-24
2019-04-02
2020-04-06
2024-02-26
Counter({29: 3224, 28: 2087, 27: 539, 30: 430})
[28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 27, 27, 27, 27, 27, 27, 27, 27, 27, 

<!-- 1. Load djia_prices_all.csv into memory as a Pandas DataFrame (df1), holding onto the dates and the FIGIs.
2. Initialise a list called windows.
3. For each date-set pair in djia_daily_constituent_figis.items():
    1. Only want 1 out of every 12 Wednesdays.
    2. Initialise a Pandas DataFrame (df2).
    3. Otherwise, for each FIGI in the set:
        1. Collect the prices in df1 for this FIGI in for each Wednesday from 52 Wednesdays prior to the current date, to 12 Wednesdays following the current date, inclusive. For Wednesdays missing from df1, accept NaN.
        2. Replace all NaNs with 0.
        3. Turn these 65 prices into 64 returns; whenever a denominator is 0, make the result 0.
        4. Add the returns to df2, labelling them with the current FIGI.
    4. Transpose (maybe) df2, maintaining a way to deduce the FIGI of each price.
    5. Extract the first 52 price arrays into training_df and the remaining 12 price arrays into test_df.
    6. Append [training_df, test_df] to windows.
4. Save windows to a Pickle file, including in the name the first Wednesday and the last Wednesday from djia_daily_constituent_figis. -->

In [11]:
import pandas as pd

prices_df = pd.read_csv(
    'djia_adj_close_yfinance_2026-03-31T19-39-17Z.csv',
    header=0,
    index_col=0,
    parse_dates=True
)

# print(prices_df.head())
# print(prices_df.columns[:5])
# print(prices_df.tail())

In [12]:
import numpy as np
from collections import defaultdict, Counter
import pickle
from mis_dro.portfolio import compute_drifted_returns_and_final_weights

num_training_weeks = 52

training_missing_freq_histogram = defaultdict(lambda: 0)
test_missing_freq_histogram = defaultdict(lambda: 0)
training_missing_freq_histogram_markowitz = defaultdict(lambda: 0)

too_many_missing_training_data = 16
too_many_missing_test_data = 4

# 6, 4
# If too_many_missing_test_data < 13, I.E. we say it's fine as long as at least 1 test datum is valid, then not all windows will have 30 stocks

windows = []
markowitz_windows = []
benchmark_for_price_weighted_portfolio_weightings = []

count = 0

all_wednesdays = sorted(date for date in djia_daily_constituent_figis if date.weekday() == 2)
for i, date in enumerate(all_wednesdays[::rebalance_period_weeks]):

    pd_date = pd.to_datetime(date)
    start = pd_date - pd.DateOffset(weeks=num_training_weeks)
    end = pd_date + pd.DateOffset(weeks=rebalance_period_weeks)
    window_wednesdays = pd.date_range(start, end, freq='W-WED')

    training_df = pd.DataFrame()
    test_df = pd.DataFrame()

    training_df_markowitz = pd.DataFrame()

    last_dacp_before_rebalance = pd.DataFrame()

    for figi in djia_daily_constituent_figis[date]:

        window_figi_prices = prices_df[figi].reindex(window_wednesdays)

        denominator = window_figi_prices.shift(1)
        wednesday_returns = (window_figi_prices - denominator) / denominator
        wednesday_returns = wednesday_returns.iloc[1:]

        training_wednesday_returns = wednesday_returns.iloc[:num_training_weeks]
        test_wednesday_returns = wednesday_returns.iloc[num_training_weeks:]

        num_training_missing = training_wednesday_returns.isna().sum() + np.isinf(training_wednesday_returns).sum()
        num_test_missing = test_wednesday_returns.isna().sum() + np.isinf(test_wednesday_returns).sum()

        training_missing_freq_histogram[num_training_missing] += 1
        test_missing_freq_histogram[num_test_missing] += 1

        if num_training_missing >= too_many_missing_training_data:
            continue

        if num_test_missing >= too_many_missing_test_data:
            continue

        # NOTE: the below is what I used for the experiment with 4 week rebalance
        # if num_training_nans >= 11 or num_test_nans == 4:
        #     continue

        # NOTE: not factoring training returns into interpolation for test returns or vice versa because don't want to synthesise test data using training data due to bias
        training_wednesday_returns = training_wednesday_returns.replace([np.inf, -np.inf], np.nan)
        test_wednesday_returns = test_wednesday_returns.replace([np.inf, -np.inf], np.nan)

        for series in (training_wednesday_returns, test_wednesday_returns):
            series.interpolate(method="time", limit_direction="both", inplace=True)

        training_df[f"{figi}_{i}"] = training_wednesday_returns.values
        test_df[f"{figi}_{i}"] = test_wednesday_returns.values

        # NOTE: below is for the parallel Markowitz windows

        denominator_markowitz = window_figi_prices.shift(13)
        thirteen_wednesday_returns = (window_figi_prices - denominator_markowitz) / denominator_markowitz
        thirteen_wednesday_returns = thirteen_wednesday_returns.iloc[13:]
        training_thirteen_wednesday_returns = thirteen_wednesday_returns.iloc[:num_training_weeks-12]

        num_training_missing_markowitz = training_thirteen_wednesday_returns.isna().sum() + np.isinf(training_thirteen_wednesday_returns).sum()
        training_missing_freq_histogram_markowitz[num_training_missing_markowitz] += 1

        training_thirteen_wednesday_returns = training_thirteen_wednesday_returns.replace([np.inf, -np.inf], np.nan)
        training_thirteen_wednesday_returns.interpolate(method="time", limit_direction="both", inplace=True)

        training_df_markowitz[f"{figi}_{i}"] = training_thirteen_wednesday_returns.values

        # NOTE: for getting benchmark portfolio weightings (which are weighted based on the DJIA consituent prices, much like for the DJIA itself), using the DACP on the final Wednesday before rebalance
        last_dacp_before_rebalance[f"{figi}_{i}"] = [window_figi_prices.iloc[52]]
        
        last_dacp = window_figi_prices.iloc[52]
        if np.isnan(last_dacp) or np.isinf(last_dacp):
            count += 1
            print(count)

    price_weighted_portfolio_weighting = last_dacp_before_rebalance.iloc[0] / last_dacp_before_rebalance.iloc[0].sum()
    out_of_sample_cost_for_price_weighted_portfolio_weighting = test_df @ price_weighted_portfolio_weighting

    oos_portfolio_returns_with_weighting_drift_for_price_weighted_portfolio_weighting, drifted_price_weighted_portfolio_weighting = compute_drifted_returns_and_final_weights(test_df, price_weighted_portfolio_weighting)
    # NOTE (pwd): make oos_portfolio_returns_with_weighting_drift_for_price_weighted_portfolio_weighting the same way you calculate oos_portfolio_returns_with_weighting_drift in mis_dro/main.py
    # NOTE (pwd): make drifted_price_weighted_portfolio_weighting the same way you calculate drifted_weighting in mis_dro/main.py

    windows.append([training_df, test_df])
    markowitz_windows.append([training_df_markowitz, test_df])
    benchmark_for_price_weighted_portfolio_weightings.append({
        "weighting": price_weighted_portfolio_weighting,    # TODO: remove the appending of _{i} that you have been doing in the first place and update all files accordingly
        "out_of_sample_cost": out_of_sample_cost_for_price_weighted_portfolio_weighting,

        "oos_portfolio_returns_with_weighting_drift": oos_portfolio_returns_with_weighting_drift_for_price_weighted_portfolio_weighting,
        "drifted_weighting": drifted_price_weighted_portfolio_weighting
        # NOTE (pwd): set "oos_portfolio_returns_with_weighting_drift" to oos_portfolio_returns_with_weighting_drift_for_price_weighted_portfolio_weighting,
        # NOTE (pwd): set "drifted_weighting" to drifted_price_weighted_portfolio_weighting
    })

# print(training_missing_freq_histogram)
# print(test_missing_freq_histogram)
# print(training_missing_freq_histogram_markowitz)    # NOTE: this shows that, for the Markowitz windows, at most, in any window, a stock has 8 (out of 40) missing values, so no need to discard any stocks (taking 30% as the limit), fortunately--just got to interpolate

if overwrite_previous_preprocessed_data:

    last_wed = date

    _fname = f"windows_rebalance_dates_{all_wednesdays[0].strftime('%Y%m%d')}_to_{last_wed.strftime('%Y%m%d')}_inclusive_every_{rebalance_period_weeks}_weeks"

    fname = f"{_fname}.pkl"

    with open(fname, 'wb') as f:
        pickle.dump(windows, f)

    fname_markowitz = f"{_fname}_markowitz.pkl"

    with open(fname_markowitz, 'wb') as f:
        pickle.dump(markowitz_windows, f)

    fname_benchmark_for_price_weighted_portfolio_weightings = f"{_fname}_benchmark_for_price_weighted_portfolio_weightings.pkl"

    with open(fname_benchmark_for_price_weighted_portfolio_weightings, 'wb') as f:
        pickle.dump(benchmark_for_price_weighted_portfolio_weightings, f)

num_dims_per_window = []
for window in windows:
    try:
        num_dims = len(window[0].iloc[0])
    except:
        num_dims = 0
    num_dims_per_window.append(num_dims)
print(Counter(num_dims_per_window))

Counter({29: 36, 28: 20, 27: 9, 30: 5})
